# Translate gene IDs

## Mygene

In [ ]:
import mygene

# Initialize the MyGene.info client
mg = mygene.MyGeneInfo()

# Query example: Convert Ensembl Gene IDs to other types
query = ['ENSG00000141510', 'ENSG00000139618']  # Example Ensembl Gene IDs (TP53, BRCA2)

# Query MyGene.info for mappings
print("Querying MyGene.info...")
results = mg.querymany(
    query, 
    scopes='ensembl.gene', 
    fields='symbol,entrezgene,uniprot', 
    species='human'
)

# Display the results
print("\nResults from MyGene.info:")
for res in results:
    print(f"Ensembl ID: {res.get('query')}")
    print(f"  Symbol: {res.get('symbol')}")
    print(f"  Entrez Gene ID: {res.get('entrezgene')}")
    print(f"  UniProt ID: {res.get('uniprot', {}).get('Swiss-Prot')}")
    print("-" * 40)


### Example with adata file

In [ ]:
import pandas as pd
import numpy as np
import anndata as ad
import mygene

In [ ]:
# Load the adata file
adata = ad.read_h5ad("/storage/users/data/PANC/H5AD_file/adata_filtered_no2D_hvg_clust_time_pub.h5ad")
adata.var_names

In [ ]:
# Extract Ensembl Gene IDs
ensembl_ids = adata.var_names.tolist()

# Initialize MyGene.info client
mg = mygene.MyGeneInfo()

# Query MyGene.info for mappings
print("Querying MyGene.info...")
results = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol,entrezgene,uniprot",
    species="human"
)

# Create dictionaries for mappings
ensembl_to_symbol = {}
ensembl_to_entrez = {}
ensembl_to_uniprot = {}

# Populate dictionaries
print("Processing results...")
for res in results:
    ensembl_id = res.get("query")
    if "notfound" in res:
        continue  # Skip if not found
    if ensembl_id:
        ensembl_to_symbol[ensembl_id] = res.get("symbol")
        ensembl_to_entrez[ensembl_id] = res.get("entrezgene")
        uniprot = res.get("uniprot", {}).get("Swiss-Prot")  # Get Swiss-Prot ID
        ensembl_to_uniprot[ensembl_id] = uniprot

# Display dictionary samples
print("\nSample mappings:")
print("Ensembl to Gene Symbol:", list(ensembl_to_symbol.items())[:15])
print("Ensembl to Entrez Gene ID:", list(ensembl_to_entrez.items())[:15])
print("Ensembl to UniProt ID:", list(ensembl_to_uniprot.items())[:15])


In [ ]:
# Extract Ensembl Gene IDs
ensembl_ids = adata.var_names.tolist()

# Initialize MyGene.info client
mg = mygene.MyGeneInfo()

# Query MyGene.info for mappings
print("Querying MyGene.info...")
results = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol,entrezgene,uniprot",
    species="human"
)

# Create dictionaries for mappings
ensembl_to_symbol = {}
ensembl_to_entrez = {}
ensembl_to_uniprot = {}

# Counters for matched and unmatched IDs
matched_count = 0
unmatched_count = 0

# Populate dictionaries
print("Processing results...")
for res in results:
    ensembl_id = res.get("query")
    if "notfound" in res:
        # If not found, use Ensembl ID as a fallback
        ensembl_to_symbol[ensembl_id] = ensembl_id
        ensembl_to_entrez[ensembl_id] = ensembl_id
        ensembl_to_uniprot[ensembl_id] = ensembl_id
        unmatched_count += 1
        continue

    # Populate dictionaries with retrieved or fallback values
    ensembl_to_symbol[ensembl_id] = res.get("symbol", ensembl_id)
    ensembl_to_entrez[ensembl_id] = res.get("entrezgene", ensembl_id)
    uniprot = res.get("uniprot", {}).get("Swiss-Prot", ensembl_id)
    ensembl_to_uniprot[ensembl_id] = uniprot
    matched_count += 1

# Display dictionary samples
print("\nSample mappings:")
print("Ensembl to Gene Symbol:", list(ensembl_to_symbol.items())[:5])
print("Ensembl to Entrez Gene ID:", list(ensembl_to_entrez.items())[:5])
print("Ensembl to UniProt ID:", list(ensembl_to_uniprot.items())[:5])

# Print summary of matches
total_ids = len(ensembl_ids)
print(f"\nSummary:")
print(f"Total Ensembl IDs: {total_ids}")
print(f"Matched IDs: {matched_count}")
print(f"Unmatched IDs: {unmatched_count}")


## Using biomart package (sometime not working)

In [ ]:
# pip install biomart

In [ ]:
from biomart import BiomartServer

# Connect to the BioMart server
#server = BiomartServer("http://ensembl.org/biomart")
server = BiomartServer("http://useast.ensembl.org/biomart")

In [ ]:
# Select the human genes dataset
dataset = server.datasets['hsapiens_gene_ensembl']

# Query BioMart to get mappings
response = dataset.search({
    'attributes': [
        'ensembl_gene_id',      # Ensembl Gene ID
        'uniprot_gn_id',        # UniProt ID
        'entrezgene_id',        # Entrez Gene ID
        'external_gene_name'    # Gene symbol
    ],
})

# Initialize dictionaries for mappings
ensembl_to_uniprot = {}
ensembl_to_entrez = {}
ensembl_to_gene_name = {}

# Parse the response
for line in response.iter_lines():
    # Decode the line and split into fields
    decoded_line = line.decode('utf-8')
    ensembl_id, uniprot_id, entrez_id, gene_name = decoded_line.split('\t')

    # Populate dictionaries
    if uniprot_id:
        ensembl_to_uniprot[ensembl_id] = uniprot_id
    if entrez_id:
        ensembl_to_entrez[ensembl_id] = entrez_id
    if gene_name:
        ensembl_to_gene_name[ensembl_id] = gene_name

# Example: Access the mappings
print(f"Ensembl to UniProt: {list(ensembl_to_uniprot.items())[:5]}")
print(f"Ensembl to Entrez: {list(ensembl_to_entrez.items())[:5]}")
print(f"Ensembl to Gene Name: {list(ensembl_to_gene_name.items())[:5]}")


## Other ID conversion tools

### Entrez

In [ ]:
from Bio import Entrez
# Set your email for NCBI Entrez queries
Entrez.email = "your_email@example.com"

# Function to search for a gene using NCBI Entrez
def search_entrez_gene(term):
    print(f"Searching Entrez for term: {term}")
    handle = Entrez.esearch(db="gene", term=term, retmax=5)
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]

# Search for TP53 gene in humans
entrez_ids = search_entrez_gene("TP53[Gene Name] AND human[Organism]")
print(f"Entrez Gene IDs for TP53: {entrez_ids}")

### Expasy and Swissport 

In [ ]:
from Bio import ExPASy
from Bio import SwissProt


# Using ExPASy and SwissProt to fetch protein details
def fetch_swissprot_record(uniprot_id):
    print(f"\nFetching SwissProt record for UniProt ID: {uniprot_id}")
    try:
        handle = ExPASy.get_sprot_raw(uniprot_id)
        record = SwissProt.read(handle)
        handle.close()
        return record
    except Exception as e:
        print(f"Error fetching record: {e}")
        return None

# Example UniProt ID for TP53
uniprot_id = "P04637"  # UniProt ID for TP53
record = fetch_swissprot_record(uniprot_id)
if record:
    print(f"Description: {record.description}")
    print(f"Gene Names: {record.gene_name}")
    print(f"Organism: {record.organism}")
    print(f"Keywords: {record.keywords}")




### PyBiomart (not working)

In [ ]:
# Using pybiomart for comprehensive ID mapping
from pybiomart import Dataset

dataset = Dataset(name="hsapiens_gene_ensembl", host="http://www.ensembl.org")
results = dataset.query(attributes=["ensembl_gene_id", "external_gene_name", "entrezgene_id", "uniprotswissprot"])
id_mapping = {
    row["Gene stable ID"]: {
        "external_gene_name": row["Gene name"],
        "entrez_gene_id": row["NCBI gene (formerly Entrezgene) ID"],
        "uniprot_id": row["UniProtKB/Swiss-Prot ID"],
    }
    for _, row in results.iterrows()
}

print("\nSample ID Mapping from BioMart:")
for ensembl_id, mappings in list(id_mapping.items())[:5]:
    print(f"{ensembl_id}: {mappings}")